In [11]:
from openai import OpenAI

openai_client = OpenAI()

In [12]:
from pydantic import BaseModel

class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

In [1]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index


reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

print(f"Indexed {len(chunked_docs)} chunks from {len(files)} documents")

Indexed 382 chunks from 95 documents


In [2]:
from openai import OpenAI
openai_client = OpenAI() 
response =  openai_client.responses.create(input='how is the weather in London today',model='gpt-4o-mini')
print(response.text)

ResponseTextConfig(format=ResponseFormatText(type='text'), verbosity='medium')


In [3]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

In [4]:
import json

instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return prompt_template.format(
        question=question,
        context=context
    )

In [5]:
def llm(user_prompt, instructions=None, model="gpt-4o-mini"):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [6]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm(prompt, instructions)

In [8]:
answer = rag('how do I implement LLM as a judge?')
print(answer)

To implement an LLM as a judge, follow these steps:

1. **Create an Evaluation Dataset**:
   - Prepare a toy Q&A dataset that includes questions, target responses (approved answers), new responses (imitated), and manual labels indicating whether the response is correct or not.
   - This helps in formulating better criteria and establishing a "ground truth" for evaluating the LLM judge.

2. **Install Necessary Libraries**:
   - Install the Evidently library using:
     ```python
     pip install evidently
     ```
   - Import the required modules:
     ```python
     import pandas as pd
     import numpy as np
     from evidently import Dataset
     from evidently import DataDefinition
     from evidently import Report
     from evidently import BinaryClassification
     from evidently.llm.templates import BinaryClassificationPromptTemplate
     ```

3. **Set Your OpenAI Key**:
   - Pass your OpenAI API key as an environment variable:
     ```python
     import os
     os.environ["OPENA

In [9]:
def llm_structured(
    user_prompt,
    output_type,
    instructions=None,
    model="gpt-4o-mini",
):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=output_type
    )

    return response.output_parsed

In [14]:
response = llm_structured(
    instructions="Extract the event information.",
    user_prompt="Alice and Bob are going to a science fair on Friday.",
    output_type=CalendarEvent,
)
print(response)

name='Science Fair' date='Friday' participants=['Alice', 'Bob']


In [15]:
class RAGResponse(BaseModel):
    answer: str
    found_answer: bool

In [16]:
RAGResponse.model_json_schema()

{'properties': {'answer': {'title': 'Answer', 'type': 'string'},
  'found_answer': {'title': 'Found Answer', 'type': 'boolean'}},
 'required': ['answer', 'found_answer'],
 'title': 'RAGResponse',
 'type': 'object'}

In [17]:
def rag_structured(query, output_type=RAGResponse):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm_structured(
        instructions=instructions,
        user_prompt=prompt,
        output_type=output_type,
    )

In [18]:
answer = rag_structured('how do I do llm evals?')

print(answer.answer[:100])
print(answer.found_answer)

To perform LLM evaluations, you can follow these steps:

1. **Install Evidently**:
   ```python
   !
True


In [19]:
answer = rag_structured('how do I install kafka on windows?')

print(answer.answer[:100])
print(answer.found_answer)

The CONTEXT does not provide information on how to install Kafka on Windows.
False


In [20]:
from typing import Optional

class RAGResponse(BaseModel):
    answer: Optional[str] = None
    found_answer: bool

In [21]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)

print(answer.answer)
print(answer.found_answer)

None
False


In [22]:
instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.

If you don't find the answer, set `answer` to None
"""

In [23]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)

print(answer.answer)
print(answer.found_answer)

None
False


In [24]:
class RAGResponse(BaseModel):
    """
    The response from the documentation RAG system

    If the answer to the question wasn't found in the database, `answer` is None
    """
    answer: Optional[str] = None
    found_answer: bool

In [26]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)

print(answer.answer)
print(answer.found_answer)

None
False


In [27]:
from typing import Literal

class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

In [28]:
RAGResponse.model_json_schema()

{'description': 'This model provides a structured answer with metadata about the response,\nincluding confidence, categorization, and follow-up suggestions.',
 'properties': {'answer': {'description': "The main answer to the user's question in markdown",
   'title': 'Answer',
   'type': 'string'},
  'found_answer': {'description': 'True if relevant information was found in the documentation',
   'title': 'Found Answer',
   'type': 'boolean'},
  'confidence': {'description': 'Confidence score from 0.0 to 1.0 indicating how certain the answer is',
   'title': 'Confidence',
   'type': 'number'},
  'confidence_explanation': {'description': 'Explanation about the confidence level',
   'title': 'Confidence Explanation',
   'type': 'string'},
  'answer_type': {'description': 'The category of the answer',
   'enum': ['how-to',
    'explanation',
    'troubleshooting',
    'comparison',
    'reference'],
   'title': 'Answer Type',
   'type': 'string'},
  'followup_questions': {'description': 'S

In [29]:
answer = rag_structured('how do I evaluate llms', RAGResponse)

In [30]:
print(answer)

answer='To evaluate LLM outputs, you can use a panel of multiple LLMs as judges. Here’s a structured approach:\n\n### Steps to Evaluate LLM Outputs\n\n1. **Set Up the Environment**:\n   - Install the necessary package:\n   ```bash\n   pip install evidently litellm\n   ```\n   - Import required components:\n   ```python\n   import pandas as pd\n   from evidently import Dataset, DataDefinition, Report\n   from evidently.presets import TextEvals\n   from evidently.descriptors import LLMEval, TestSummary\n   ```\n\n2. **Set Up Evaluator LLMs**:\n   - Pass the API keys for the LLMs to be used as judges:\n   ```python\n   import os\n   os.environ["OPENAI_API_KEY"] = "YOUR_KEY"\n   os.environ["GEMINI_API_KEY"] = "YOUR_KEY"\n   os.environ["ANTHROPIC_API_KEY"] = "YOUR_KEY"\n   ```\n\n3. **Create a Panel of LLM Judges**:\n   - Define a dataset for evaluation (e.g., user instructions and generated emails).\n   - Use a `BinaryClassificationPromptTemplate` to specify the criteria for judgment.\n   

In [31]:
from pydantic import model_validator

class AnswerNotFound(BaseModel):
    explanation: str

class AnswerResponse(BaseModel):
    """
    If answer is found, 'answer' is populated.
    If no answer is found, 'answer_not_found' is populated.
    Only one of the two fields can be set at a time. Never both or neither.
    """

    answer_not_found: Optional[AnswerNotFound] = None
    found_answer: bool
    answer: Optional[RAGResponse] = None

    @model_validator(mode="after")
    def check_consistency(self):
        if self.answer is not None and self.answer_not_found is not None:
            raise ValueError("Provide either 'answer' or 'answer_not_found', not both.")

        if self.answer is None and self.answer_not_found is None:
            raise ValueError("Provide either 'answer' or 'answer_not_found'.")

        return self

In [32]:
AnswerResponse.model_json_schema()

{'$defs': {'AnswerNotFound': {'properties': {'explanation': {'title': 'Explanation',
     'type': 'string'}},
   'required': ['explanation'],
   'title': 'AnswerNotFound',
   'type': 'object'},
  'RAGResponse': {'description': 'This model provides a structured answer with metadata about the response,\nincluding confidence, categorization, and follow-up suggestions.',
   'properties': {'answer': {'description': "The main answer to the user's question in markdown",
     'title': 'Answer',
     'type': 'string'},
    'found_answer': {'description': 'True if relevant information was found in the documentation',
     'title': 'Found Answer',
     'type': 'boolean'},
    'confidence': {'description': 'Confidence score from 0.0 to 1.0 indicating how certain the answer is',
     'title': 'Confidence',
     'type': 'number'},
    'confidence_explanation': {'description': 'Explanation about the confidence level',
     'title': 'Confidence Explanation',
     'type': 'string'},
    'answer_type': 

In [33]:
answer = rag_structured('how do I install kafka on windows?', AnswerResponse)
answer

ValidationError: 1 validation error for AnswerResponse
  Value error, Provide either 'answer' or 'answer_not_found'. [type=value_error, input_value={'answer_not_found': None...: False, 'answer': None}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

In [34]:
answer = rag_structured('how do I run llm evals?', AnswerResponse)
answer

AnswerResponse(answer_not_found=None, found_answer=True, answer=RAGResponse(answer='## Running LLM Evaluations\n\nTo run LLM evaluations, follow the instructions below:\n\n1. **Installation**: First, ensure that you have the Evidently library installed with LLM support:\n   ```bash\n   pip install evidently[llm]\n   ```\n\n2. **Import Required Modules**:\n   ```python\n   import pandas as pd\n   from evidently.future.datasets import Dataset\n   from evidently.future.datasets import DataDefinition\n   from evidently.future.report import Report\n   from evidently.future.presets import TextEvals\n   from evidently.ui.workspace.cloud import CloudWorkspace\n   ```\n\n3. **Connect to Evidently Cloud**: Replace with your actual token:\n   ```python\n   ws = CloudWorkspace(token="YOUR_API_TOKEN", url="https://app.evidently.cloud")\n   ```\n\n4. **Create Your Project**:\n   ```python\n   project = ws.create_project("Project Name", org_id="YOUR_ORG_ID")\n   project.description = "Your project de

In [35]:
from pydantic import ValidationError

try:
    AnswerResponse()
except ValidationError as e:
    print("Validation error:")
    print(e)

Validation error:
1 validation error for AnswerResponse
found_answer
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
